In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential, load_model
from keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Dropout, Flatten, Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
save_best = ModelCheckpoint('./best_model.keras', monitor='val_loss', save_best_only=True)

es = EarlyStopping(monitor = "val_loss", mode = "min", patience = 10, restore_best_weights=True, verbose=1)

EPOCHS = 1000


In [ ]:
root_path = "/usr/share/MSI25/tumor"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    root_path,
    label_mode="categorical",
    batch_size=32,
    image_size=(100, 100),
    validation_split=0.2,
    subset="training",
    color_mode = "grayscale",
    seed=123,
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    root_path,
    label_mode="categorical",
    batch_size=32,
    image_size=(100, 100),
    validation_split=0.2,
    subset="validation",
    color_mode = "grayscale",
    seed=123,
    shuffle=True
)
class_names = train_dataset.class_names   # tutaj, przed .map
print(class_names)

# odpowiednik rescale=1./255 - piksele z zakresu 0-255 na 0-1
train_dataset = train_dataset.map(lambda x, y: (x / 255.0, y))
validation_dataset = validation_dataset.map(lambda x, y: (x / 255.0, y))


In [ ]:
plt.figure(figsize=(10,10))
img, label = next(iter(validation_dataset))
img = img.numpy()
label = label.numpy()
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.imshow(img[i])
    plt.axis('off')
    if label[i][0]==1:
        plt.title("tumor")
    else:
        plt.title("healthy")
            

In [ ]:
reloaded_model = load_model("best_model.keras")
reloaded_model.summary()

In [ ]:
img, label = next(iter(validation_dataset))
img = img.numpy()
label = label.numpy()
y_pred = reloaded_model.predict(img)

In [ ]:
plt.figure(figsize=(10,10))
for i in range(1,26):
    plt.subplot(5,5,i)
    plt.imshow(img[i-1], cmap="gray")
    plt.axis('off')
    if label[i][0]==1:
        title = "tumor"
    else:
        title = "healthy"
        
    if np.round(y_pred[i])[0] == label[i][0]:
        plt.title(title, color = 'green')
    else:
        plt.title(title, color = 'red')        

In [ ]:
def grad_cam(model, obraz):
    # numer ostatniej warstwy konwolucyjnej w modelu
    ostatnia_konw = max(i for i, w in enumerate(model.layers) if isinstance(w, Conv2D))

    x = tf.convert_to_tensor(obraz[np.newaxis, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        for i, warstwa in enumerate(model.layers):    # przepuszczamy obraz warstwa po warstwie
            x = warstwa(x, training=False)            # ZMIANA: wyłącza augmentację i Dropout
            if i == ostatnia_konw:
                mapy_cech = x                         # zapamiętujemy mapy cech z ostatniej konwolucji
                tape.watch(mapy_cech)
        klasa = tf.argmax(x[0])                       # klasa wybrana przez sieć
        wynik = x[0, klasa]                           # pewność sieci dla tej klasy

    grad = tape.gradient(wynik, mapy_cech)[0]         # jak każda mapa cech wpływa na wynik
    wagi = tf.reduce_mean(grad, axis=(0, 1))          # jedna waga (ważność) dla każdego filtra
    cam = tf.nn.relu(tf.reduce_sum(mapy_cech[0] * wagi, axis=-1))  # suma map cech razy ich ważność
    cam = cam / (tf.reduce_max(cam) + 1e-8)           # skalowanie do zakresu 0-1
    cam = tf.image.resize(cam[..., tf.newaxis], obraz.shape[:2])[..., 0]  # powiększenie do rozmiaru zdjęcia
    return cam.numpy(), int(klasa)

img, label = next(iter(validation_dataset))
img = img.numpy()
label = label.numpy()

plt.figure(figsize=(12, 5))
for i in range(6):
    cam, klasa = grad_cam(reloaded_model, img[i])
    prawdziwa = np.argmax(label[i])
    kolor = 'green' if klasa == prawdziwa else 'red'

    plt.subplot(2, 6, i + 1)
    plt.imshow(img[i][:, :, 0], cmap='gray')
    plt.title(class_names[klasa], color=kolor)
    plt.axis('off')

    plt.subplot(2, 6, i + 7)
    # plt.imshow(img[i][:, :, 0], cmap='gray')
    plt.imshow(cam, cmap='jet', alpha=0.4)
    plt.axis('off')